In [1]:
import pandas as pd
from scipy import stats
import numpy as np
from pandas.api.types import CategoricalDtype
import re
from helpers import get_renamed_blocks, get_renamed_blocks3, get_renamed_blocks4

In [2]:
names_dict4 = {'Datensatztyp*': 'datatype', 'EinheitMastrNummer': "blockid", 'Anlagenbetreiber': "company", 'Anzeigename': "plantname", 'Postleitzahl': "plz", 'Ort': "place", 'Straße': "street", 'Hausnummer': "streetnum", 'Bundesland': "federalstate", 'Land': 'nation', 'Jahr_Inbetriebnahme': 'initialopYear', 'Jahr_Stilllegung**': 'endop', 'Kraftwerksstatus': "state", 'Energieträger': "energysource", 'Hauptbrennstoff': "e2", 'Waermeauskopplung_KWK': "chp", 'Erneuerbarer_Energietraeger': "eeg", 'Bruttoleistung_MW': "grosspower", 'Nettonennleistung_MW': "power", 'Technologie_Stromerzeugung': "tech", 'Volleinspeisung_Teileinspeisung': "fullsupply", 'Spannungsebene': "voltagelevel", 'Anschlussnetzbetreiber': "TSO", 'Bestandteil_Grenzkraftwerk': 'border', 'Grenzkraftwerk_Nettonennleistung_MW': "gerpower"}


In [3]:
names_dict4

{'Datensatztyp*': 'datatype',
 'EinheitMastrNummer': 'blockid',
 'Anlagenbetreiber': 'company',
 'Anzeigename': 'plantname',
 'Postleitzahl': 'plz',
 'Ort': 'place',
 'Straße': 'street',
 'Hausnummer': 'streetnum',
 'Bundesland': 'federalstate',
 'Land': 'nation',
 'Jahr_Inbetriebnahme': 'initialopYear',
 'Jahr_Stilllegung**': 'endop',
 'Kraftwerksstatus': 'state',
 'Energieträger': 'energysource',
 'Hauptbrennstoff': 'e2',
 'Waermeauskopplung_KWK': 'chp',
 'Erneuerbarer_Energietraeger': 'eeg',
 'Bruttoleistung_MW': 'grosspower',
 'Nettonennleistung_MW': 'power',
 'Technologie_Stromerzeugung': 'tech',
 'Volleinspeisung_Teileinspeisung': 'fullsupply',
 'Spannungsebene': 'voltagelevel',
 'Anschlussnetzbetreiber': 'TSO',
 'Bestandteil_Grenzkraftwerk': 'border',
 'Grenzkraftwerk_Nettonennleistung_MW': 'gerpower'}

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
#STATES = ['in Betrieb', 'Gesetzlich an Stilllegung gehindert', 'Netzreserve',  'Sicherheitsbereitschaft', 'Sonderfall', 'vorläufig stillgelegt', 'stillgelegt', 'Kohlestromvermarktungsverbot']
#STATES = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'Strommarktrückkehr', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt KVBG', 'stillgelegt']
STATES = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'bnBm', 'KVBG','stillgelegt', 'vorläufig stillgelegt']
ACTIVE_STATES = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'bnBm', 'vorläufig stillgelegt']
ENERGIES = ["Kernenergie", "Braunkohle", "Steinkohle", "Erdgas", "Mineralölprodukte", "Abfall", "Biomasse", ""]

In [6]:
bm = pd.read_csv("../basic/inspire_prtr_mapper.csv", engine="python")
#plr = pd.read_excel("../data/Kraftwerksliste.xlsx", sheet_name=1, skiprows=10, decimal=',')
pl0 = pd.read_excel("../data/Kraftwerksliste.xlsx", skiprows=7, decimal=',')
prtr = pd.read_excel("../data/2023-12-08_PRTR-Deutschland_Freisetzungen.xlsx")

In [7]:
list(pl0)

['Datensatztyp*',
 'EinheitMastrNummer',
 'Anlagenbetreiber',
 'Anzeigename',
 'Postleitzahl',
 'Ort',
 'Strasse',
 'Hausnummer',
 'Bundesland',
 'Land',
 'Jahr_Inbetriebnahme',
 'Jahr_Stilllegung**',
 'Kraftwerksstatus',
 'Energietraeger',
 'Hauptbrennstoff',
 'Waermeauskopplung_KWK',
 'Erneuerbarer_Energietraeger',
 'Bruttoleistung_MW',
 'Nettonennleistung_MW',
 'Technologie_Stromerzeugung',
 'Volleinspeisung_Teileinspeisung',
 'Spannungsebene',
 'Anschlussnetzbetreiber',
 'Bestandteil_Grenzkraftwerk',
 'Grenzkraftwerk_Nettonennleistung_MW']

In [8]:
bm['PRTR_Kennnummer'] = bm['PRTR_Kennnummer'].apply(lambda x: str(x).replace('/', '_'))
bm['InspireID_Betrieb'] = bm['InspireID_Betrieb'].apply(lambda x: str(x).replace('/', '_'))

In [9]:
#bm.loc[bm.plantid.str.contains("_")]

In [10]:
pl0_bak = pl0.copy()

In [11]:
list(pl0_bak)

['Datensatztyp*',
 'EinheitMastrNummer',
 'Anlagenbetreiber',
 'Anzeigename',
 'Postleitzahl',
 'Ort',
 'Strasse',
 'Hausnummer',
 'Bundesland',
 'Land',
 'Jahr_Inbetriebnahme',
 'Jahr_Stilllegung**',
 'Kraftwerksstatus',
 'Energietraeger',
 'Hauptbrennstoff',
 'Waermeauskopplung_KWK',
 'Erneuerbarer_Energietraeger',
 'Bruttoleistung_MW',
 'Nettonennleistung_MW',
 'Technologie_Stromerzeugung',
 'Volleinspeisung_Teileinspeisung',
 'Spannungsebene',
 'Anschlussnetzbetreiber',
 'Bestandteil_Grenzkraftwerk',
 'Grenzkraftwerk_Nettonennleistung_MW']

In [12]:
cols = [14,20,23,24]
plt = get_renamed_blocks4(pl0)
plq = plt.drop(plt.columns[cols],axis=1)

In [13]:
#plr0a.sort_values('power')

In [14]:
#list(plq)
#list(plr0b)

In [15]:
list(plq)

['datatype',
 'blockid',
 'company',
 'plantname',
 'plz',
 'place',
 'street',
 'streetnum',
 'federalstate',
 'nation',
 'initialopYear',
 'endop',
 'state',
 'energysource',
 'chp',
 'eeg',
 'grosspower',
 'power',
 'tech',
 'voltagelevel',
 'TSO']

In [16]:
plq

,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO
0,Einzelanlage,SEE900600559253,Rheinkraftwerk Albbruck-Dogern AG,RADAG Wehrkraftwerk,79804,Dogern,Zollstraße,NaN,BadenWuerttemberg,Deutschland,2009.0,NaN,InBetrieb,Wasser,Nein,Ja,28.450,24.000,Laufwasseranlage,Hochspannung,Amprion GmbH (SNB976890256486)
1,Einzelanlage,SEE900838448145,Zweckverband Müllverwertung Schwandorf,MKW SAD Turbine 3,92421,Schwandorf,Alustraße,7,Bayern,Deutschland,1982.0,NaN,InBetrieb,Abfall,Ja,Nein,10.000,10.000,KondensationsmaschinemitEntnahme,Hochspannung,Bayernwerk Netz GmbH (SNB940352624434)
2,Einzelanlage,SEE901246020381,TEAG Thüringer Energie AG,Gasmotor Bad Salzungen,36433,Bad Salzungen,Langenfelder Straße,82,Thueringen,Deutschland,2018.0,NaN,InBetrieb,Erdgas,Ja,Nein,10.051,10.003,Verbrennungsmotor,Mittelspannung,TEN Thüringer Energienetze GmbH & Co. KG (SNB9...
3,Einzelanlage,SEE901752522324,EnBW Energie Baden-Württemberg AG,Rheinhafen-Dampfkraftwerk RDK 8 Wasserturbine,76189,Karlsruhe,Fettweisstraße,42,BadenWuerttemberg,Deutschland,2014.0,NaN,InBetrieb,Wasser,Nein,Ja,1.800,1.800,Abwasserkraftanlage,Mittelspannung,Netze BW GmbH (SNB948311994307)
4,Einzelanlage,SEE905721866655,RWE Power AG,Fab Frechen,50226,Frechen,Ludwigstraße,NaN,NordrheinWestfalen,Deutschland,1962.0,NaN,InBetrieb,Braunkohle,Ja,Nein,82.000,56.000,KondensationsmaschinemitEntnahme,Hochspannung,Westnetz GmbH (SNB921897286493)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2609,stillgelegte Anlagen,BNA0705,NaN,Niederaußem D,50129,Bergheim,NaN,NaN,NordrheinWestfalen,Deutschland,1968.0,2021.0,endgültig stillgelegt nach KVBG,Braunkohle,Ja,Nein,320.000,297.000,NaN,NaN,NaN
2610,stillgelegte Anlagen,BNA0439,NaN,Buschhaus,38350,Helmstedt,NaN,NaN,Niedersachsen,Deutschland,1985.0,2020.0,endgültig stillgelegt nach KVBG,Braunkohle,Nein,Nein,390.000,352.000,NaN,NaN,NaN
2611,stillgelegte Anlagen,BNA1183,NaN,Heizkraftwerk Merheim,51109,Köln,NaN,NaN,NordrheinWestfalen,Deutschland,2001.0,2020.0,endgültig stillgelegt nach KVBG,Erdgas,Ja,Nein,17.400,15.800,NaN,NaN,NaN
2612,stillgelegte Anlagen,BNA0199,NaN,Dormagen,41539,Dormagen,NaN,NaN,NordrheinWestfalen,Deutschland,2000.0,2021.0,endgültig stillgelegt nach KVBG,Erdgas,Ja,Nein,597.000,24.000,NaN,NaN,NaN


In [17]:
plq0a = plq[~plq['datatype'].isin(["Kleinanlagen_aggregiert"])] # remove non-conventional facilities
plq0b = plq0a.sort_values(['power'], ascending=False)
plq0c = plq0b.dropna(subset="datatype")

In [18]:
#plq0b

In [19]:
plq0c

,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO
1968,stillgelegte Anlagen,SEE943690268513,NaN,Isar 2,84051,Essenbach,Dammstraße,NaN,Bayern,Deutschland,1988.0,2023.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Kernenergie,Nein,Nein,1485.0,1410.0,Druckwasserreaktor,NaN,NaN
2030,stillgelegte Anlagen,SEE951462745445,NaN,Brokdorf,25576,Brokdorf,Osterende,NaN,SchleswigHolstein,Deutschland,1986.0,2022.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Kernenergie,Nein,Nein,1480.0,1410.0,Druckwasserreaktor,NaN,NaN
2608,stillgelegte Anlagen,BNA0802,NaN,Kernkraftwerk Philippsburg 2,76661,Philippsburg,Rheinschanzinsel,NaN,BadenWuerttemberg,Deutschland,1985.0,2020.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Kernenergie,Nein,Nein,1468.0,1402.0,NaN,NaN,NaN
537,stillgelegte Anlagen,SEE930752846949,PreussenElektra,Grohnde,31860,Emmerthal,Kraftwerksgelände,NaN,Niedersachsen,Deutschland,1984.0,2022.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Kernenergie,Nein,Nein,1430.0,1360.0,Druckwasserreaktor,"Hochspannung, Hoechstspannung",Westfalen Weser Netz GmbH (SNB929881052512)
2028,stillgelegte Anlagen,SEE944567587799,NaN,Emsland A,49811,Lingen,Am Hilgenberg,2,Niedersachsen,Deutschland,1988.0,2023.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Kernenergie,Nein,Nein,1406.0,1336.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2282,stillgelegte Anlagen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BadenWuerttemberg,Deutschland,NaN,NaN,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Abfall,NaN,Nein,0.0,0.0,NaN,NaN,NaN
2295,stillgelegte Anlagen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SchleswigHolstein,Deutschland,NaN,NaN,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Abfall,NaN,Nein,0.0,0.0,NaN,NaN,NaN
2355,stillgelegte Anlagen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hessen,Deutschland,NaN,NaN,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Waerme,NaN,Nein,0.0,0.0,NaN,NaN,NaN
2362,stillgelegte Anlagen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SachsenAnhalt,Deutschland,NaN,NaN,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Wasser,NaN,Ja,0.0,0.0,NaN,NaN,NaN


In [20]:
def repl_fedstate(x):
    if x == 'BadenWuerttemberg':
        return 'Baden-Württemberg'
    elif x == 'MecklenburgVorpommern':
        return 'Mecklenburg-Vorpommern'
    elif x == 'NordrheinWestfalen':
        return 'Nordrhein-Westfalen'
    elif x == 'RheinlandPfalz':
        return 'Rheinland-Pfalz'
    elif x == 'SachsenAnhalt':
        return 'Sachsen-Anhalt'
    elif x == 'SchleswigHolstein':
        return 'Schleswig-Holstein'
    elif x == 'Thueringen':
        return 'Thüringen'
    else:
        return x

In [21]:
pl_combined = plq0c
pl_combined['energysource'] = pl_combined['energysource'].apply(lambda x: str(x).replace('Wärme', 'Erdgas'))
pl_combined['energysource'] = pl_combined['energysource'].apply(lambda x: str(x).replace('Mineraloelprodukte', 'Mineralölprodukte'))
pl_combined['federalstate'] = pl_combined['federalstate'].apply(lambda x: repl_fedstate(x))

In [22]:
#pl_combined

In [23]:
pl_combined.loc[pl_combined.blockid == "BNA0711"]

,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO
2505,stillgelegte Anlagen,BNA0711,NaN,Niederaußem,50129,Bergheim,NaN,NaN,Nordrhein-Westfalen,Deutschland,1963.0,2012.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Braunkohle,Nein,Nein,140.0,125.0,NaN,NaN,NaN


In [24]:
pl_combined.shape

(2323, 21)

In [25]:
pl0 = pl_combined.copy()

In [26]:
'''
pl_debug = pl0.copy()
pl_debug['initialop'] = pd.to_datetime(pl_debug['initialop'], errors='coerce')
pl_debug2 = pl_debug.copy()
pl1 = pl0.loc[pl0["energysource"].isin(ENERGIES)]
pl_debug = pl0.copy()
pl_debug['initialop'] = pd.to_datetime(pl_debug['initialop'], errors='coerce')
pl_debug2 = pl_debug.copy()
'''

'\npl_debug = pl0.copy()\npl_debug[\'initialop\'] = pd.to_datetime(pl_debug[\'initialop\'], errors=\'coerce\')\npl_debug2 = pl_debug.copy()\npl1 = pl0.loc[pl0["energysource"].isin(ENERGIES)]\npl_debug = pl0.copy()\npl_debug[\'initialop\'] = pd.to_datetime(pl_debug[\'initialop\'], errors=\'coerce\')\npl_debug2 = pl_debug.copy()\n'

In [27]:
pl0.groupby("federalstate").count()

,datatype,blockid,company,plantname,plz,place,street,streetnum,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO
federalstate,,,,,,,,,,,,,,,,,,,,
Baden-Württemberg,265,252,229,252,252,252,250,220,265,252,25,265,265,252,265,265,265,240,229,229
Bayern,445,431,414,431,431,431,412,377,445,430,23,445,445,431,445,445,445,420,414,414
Berlin,58,50,37,50,50,50,50,45,58,50,13,58,58,50,58,58,58,45,37,37
Brandenburg,88,77,67,77,77,77,72,66,88,77,10,88,88,77,88,88,88,74,67,67
Bremen,33,29,23,29,29,29,29,26,33,29,6,33,33,29,33,33,33,26,23,23
Hamburg,26,20,16,20,20,20,20,19,26,20,4,26,26,20,26,26,26,19,16,16
Hessen,136,124,118,124,124,124,118,110,136,124,6,136,136,124,136,136,136,119,118,118
Mecklenburg-Vorpommern,34,27,27,27,27,27,22,22,34,27,1,34,34,27,34,34,34,27,27,27
Niedersachsen,180,170,153,170,170,170,169,160,180,170,23,180,180,170,180,180,180,166,153,153


In [28]:
pl0.groupby("energysource").count()

,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,chp,eeg,grosspower,power,tech,voltagelevel,TSO
energysource,,,,,,,,,,,,,,,,,,,,
Abfall,120,114,109,114,114,114,114,112,120,120,114,5,120,114,120,120,120,112,109,109
Batteriespeicher,136,125,125,125,125,125,78,63,136,136,125,0,136,125,136,136,136,125,125,125
Biomasse,129,115,110,115,115,115,109,108,129,129,115,7,129,115,129,129,129,115,110,110
Braunkohle,85,78,39,78,78,78,59,44,85,85,78,43,85,78,85,85,85,58,39,39
Erdgas,904,888,826,888,888,888,857,829,904,904,887,72,904,888,904,904,904,850,826,826
Geothermie,1,0,0,0,0,0,0,0,1,1,0,0,1,0,1,1,1,0,0,0
Grubengas,7,1,0,1,1,1,1,1,7,7,1,1,7,1,7,7,7,1,0,0
Kernenergie,9,9,1,9,9,9,7,2,9,9,9,9,9,9,9,9,9,4,1,1
Mineralölprodukte,128,113,88,113,113,113,113,104,128,128,113,31,128,113,128,128,128,104,88,88


In [29]:
pl1 = pl0.loc[pl0["energysource"].isin(ENERGIES)]

In [30]:
pl1.groupby("energysource").count()

,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,chp,eeg,grosspower,power,tech,voltagelevel,TSO
energysource,,,,,,,,,,,,,,,,,,,,
Abfall,120,114,109,114,114,114,114,112,120,120,114,5,120,114,120,120,120,112,109,109
Biomasse,129,115,110,115,115,115,109,108,129,129,115,7,129,115,129,129,129,115,110,110
Braunkohle,85,78,39,78,78,78,59,44,85,85,78,43,85,78,85,85,85,58,39,39
Erdgas,904,888,826,888,888,888,857,829,904,904,887,72,904,888,904,904,904,850,826,826
Kernenergie,9,9,1,9,9,9,7,2,9,9,9,9,9,9,9,9,9,4,1,1
Mineralölprodukte,128,113,88,113,113,113,113,104,128,128,113,31,128,113,128,128,128,104,88,88
Steinkohle,139,134,57,134,134,134,112,90,139,139,134,82,139,134,139,139,139,91,57,57


In [31]:
#pl1

In [32]:
pl0.loc[pl0["blockid"] == "BNA0645"]

,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO


In [33]:
#pl_debug2.groupby("state").count()

In [34]:
#pl_debug2.groupby("initialop").count()

In [35]:
pl1['state'] = pl1['state'].fillna("stillgelegt")

In [36]:
pl1.loc[pl1.blockid == "SEE966349705634"]

,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO
959,Einzelanlage,SEE966349705634,EnBW Energie Baden-Württemberg AG,Heizkraftwerk Stuttgart-Münster GT 88,70376,Stuttgart,Voltastraße,45,Baden-Württemberg,Deutschland,2025.0,NaN,InBetrieb,Erdgas,Ja,Nein,61.5,61.0,GasturbinenmitAbhitzekessel,Hochspannung,Stuttgart Netze GmbH (SNB947592865054)


In [37]:
#pl_debug2['initialop'] = pl_debug.initialop.dt.year

In [38]:
#pl_debug['initialop'] = pl_debug['initialop'].to_datetime()

In [39]:
#pl_debug2.groupby("state").count()

In [40]:
#pl_debug2.groupby("initialop").count()

In [41]:
pl1['state'] = pl1['state'].fillna("stillgelegt")

In [42]:
#pl0

In [43]:
#bm

In [44]:
#pl0.groupby("Energieträger").count()

In [45]:
#list(plt)

In [46]:
pl1['state'] = pl1['state'].fillna("stillgelegt")

In [47]:
pl1.loc[pl1.blockid == ""]

,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO


In [48]:
plt.loc[plt.blockid == "BNA0711"]

,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,e2,chp,eeg,grosspower,power,tech,fullsupply,voltagelevel,TSO,border,gerpower
2505,stillgelegte Anlagen,BNA0711,NaN,Niederaußem,50129,Bergheim,NaN,NaN,NordrheinWestfalen,Deutschland,1963.0,2012.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Braunkohle,NaN,Nein,Nein,140.0,125.0,NaN,NaN,NaN,NaN,Nein,NaN


In [49]:
plt = pl1.dropna(subset=["blockid"])

In [50]:
plt = plt.astype({"energysource": 'category'})

In [51]:
plt.loc[plt.blockid == "BNA0711"]

,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO
2505,stillgelegte Anlagen,BNA0711,NaN,Niederaußem,50129,Bergheim,NaN,NaN,Nordrhein-Westfalen,Deutschland,1963.0,2012.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Braunkohle,Nein,Nein,140.0,125.0,NaN,NaN,NaN


In [52]:
pl2 = pl1.copy()

In [53]:
def to_year(x):
    dt = pd.to_datetime(x)
    return dt.year

In [54]:
def to_year2(x):
    if pd.isna(x):
        return x
    elif isinstance(x, float):
        return x
    elif isinstance(x, int):
        return x
    else:
        ts = pd.Timestamp(ts_input=x)
        #print(ts)
        return int(ts.year)

In [55]:
#blocks.loc[blocks.Energieträger == "Kernenergie"]

In [56]:
pl2

,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO
1968,stillgelegte Anlagen,SEE943690268513,NaN,Isar 2,84051,Essenbach,Dammstraße,NaN,Bayern,Deutschland,1988.0,2023.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Kernenergie,Nein,Nein,1485.00,1410.00,Druckwasserreaktor,NaN,NaN
2030,stillgelegte Anlagen,SEE951462745445,NaN,Brokdorf,25576,Brokdorf,Osterende,NaN,Schleswig-Holstein,Deutschland,1986.0,2022.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Kernenergie,Nein,Nein,1480.00,1410.00,Druckwasserreaktor,NaN,NaN
2608,stillgelegte Anlagen,BNA0802,NaN,Kernkraftwerk Philippsburg 2,76661,Philippsburg,Rheinschanzinsel,NaN,Baden-Württemberg,Deutschland,1985.0,2020.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Kernenergie,Nein,Nein,1468.00,1402.00,NaN,NaN,NaN
537,stillgelegte Anlagen,SEE930752846949,PreussenElektra,Grohnde,31860,Emmerthal,Kraftwerksgelände,NaN,Niedersachsen,Deutschland,1984.0,2022.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Kernenergie,Nein,Nein,1430.00,1360.00,Druckwasserreaktor,"Hochspannung, Hoechstspannung",Westfalen Weser Netz GmbH (SNB929881052512)
2028,stillgelegte Anlagen,SEE944567587799,NaN,Emsland A,49811,Lingen,Am Hilgenberg,2,Niedersachsen,Deutschland,1988.0,2023.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Kernenergie,Nein,Nein,1406.00,1336.00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2345,stillgelegte Anlagen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Berlin,Deutschland,NaN,NaN,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Mineralölprodukte,NaN,Nein,0.03,0.03,NaN,NaN,NaN
2175,stillgelegte Anlagen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hamburg,Deutschland,NaN,NaN,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Mineralölprodukte,NaN,Nein,0.01,0.01,NaN,NaN,NaN
2178,stillgelegte Anlagen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Mecklenburg-Vorpommern,Deutschland,NaN,NaN,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Mineralölprodukte,NaN,Nein,0.00,0.00,NaN,NaN,NaN
2282,stillgelegte Anlagen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Baden-Württemberg,Deutschland,NaN,NaN,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Abfall,NaN,Nein,0.00,0.00,NaN,NaN,NaN


In [57]:
'''
pl2.loc[pl2.blockid == 'BNA0861a', 'initialop'] = 2012
pl2.loc[pl2.blockid == 'BNA1334', 'initialop'] = 2002
pl2.loc[pl2.blockid == 'BNA1141', 'initialop'] = 1970
pl2.loc[pl2.blockid == 'BNA0418', 'initialop'] = 2013
pl2.loc[pl2.blockid == 'BNA1499', 'initialop'] = 1951
pl2.loc[pl2.blockid == 'BNA1502', 'initialop'] = 2013
pl2.loc[pl2.blockid == 'BNA1500', 'initialop'] = 1990
pl2.loc[pl2.blockid == 'BNA1498', 'initialop'] = 1953
pl2.loc[pl2.blockid == 'BNA1260', 'initialop'] = 2013
pl2.loc[pl2.blockid == 'BNA1056', 'initialop'] = 2006
pl2.loc[pl2.blockid == 'BNA1114', 'initialop'] = 2012



pl2.loc[pl2.blockid == 'BNA0413b', 'initialop'] = 2014 # Westfalen D

pl2.loc[pl2.blockid == 'BNA0355', 'initialop'] = 1981 # Grafenrheinfeld
'''

#pl2['Nettoleistung'] = pl2['Nettoleistung'].str.replace('\n','')
#pl2['Nettoleistung'] = pl2[pd.to_numeric(pl2["Nettoleistung"])]
#pl2['Nettoleistung'] = pl2['Nettoleistung'].str.replace(';','.')

"\npl2.loc[pl2.blockid == 'BNA0861a', 'initialop'] = 2012\npl2.loc[pl2.blockid == 'BNA1334', 'initialop'] = 2002\npl2.loc[pl2.blockid == 'BNA1141', 'initialop'] = 1970\npl2.loc[pl2.blockid == 'BNA0418', 'initialop'] = 2013\npl2.loc[pl2.blockid == 'BNA1499', 'initialop'] = 1951\npl2.loc[pl2.blockid == 'BNA1502', 'initialop'] = 2013\npl2.loc[pl2.blockid == 'BNA1500', 'initialop'] = 1990\npl2.loc[pl2.blockid == 'BNA1498', 'initialop'] = 1953\npl2.loc[pl2.blockid == 'BNA1260', 'initialop'] = 2013\npl2.loc[pl2.blockid == 'BNA1056', 'initialop'] = 2006\npl2.loc[pl2.blockid == 'BNA1114', 'initialop'] = 2012\n\n\n\npl2.loc[pl2.blockid == 'BNA0413b', 'initialop'] = 2014 # Westfalen D\n\npl2.loc[pl2.blockid == 'BNA0355', 'initialop'] = 1981 # Grafenrheinfeld\n"

In [58]:
#pl2[] = pl2[pd.to_numeric(pl2["Nettoleistung"])]
#pl2['initialop'] = pl2['initialop'].apply(to_year2)
#pl2['endop'] = pl2['endop'].apply(to_year2)
#pl2['power'] = pl2['power'].apply(pd.to_numeric)

In [59]:
pl2.loc[pl2.blockid == "BNA0711"]

,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO
2505,stillgelegte Anlagen,BNA0711,NaN,Niederaußem,50129,Bergheim,NaN,NaN,Nordrhein-Westfalen,Deutschland,1963.0,2012.0,endgültig stillgelegt ohne § 13b EnWG oder KVBG,Braunkohle,Nein,Nein,140.0,125.0,NaN,NaN,NaN


In [60]:
#bm[pl2.duplicated(['BlockID'], keep=False)].sort_values("BlockID", ascending=False)

In [61]:
#pl2[pl2.duplicated(['blockid'], keep=False)].sort_values("blockid", ascending=False)

In [62]:
#cols = [10,11,12,14,17,18,19]
#pl3 = pl2.drop(pl2.columns[cols],axis=1)

In [63]:
#pl3.groupby('Unternehmen').max()

In [64]:
#pl3

In [65]:
def fix_company(company):
    #print(company)
    company_dict = {"RWE": "RWE AG", "Vattenfall": "Vattenfall GmbH", "Uniper": "Uniper SE", "EnBW": "EnBW AG", "Steag": "Steag GmbH", "Nordzucker": "Nordzucker AG", "Lausitz Energie": "LEAG"}
    for key, value in company_dict.items():
        if key in str(company):
            return value
    return company

In [66]:
def extract_still(x):
    match = re.findall("[0-9]{4}",x)
    return match[0] if match else np.nan

In [67]:
def fix_kwk(x):
    return "Nein" if x == "nein" else "Ja" if x == "ja" else x

In [68]:
pl4 = pl2.copy()

In [69]:
list(pl2.groupby("state").count().reset_index()['state'])

['InBetrieb',
 'Kapazitätsreserve aufgrund von § 13e EnWG',
 'Netzreserve aufgrund von KVBG',
 'Netzreserve aufgrund § 13b EnWG',
 'besonderes netztechnisches Betriebsmittel',
 'endgültig stillgelegt nach KVBG',
 'endgültig stillgelegt nach § 13b EnWG',
 'endgültig stillgelegt ohne § 13b EnWG oder KVBG',
 'vorläufig Stillgelegt',
 'zeitl. gestreckte Stilllegung aufgrund § 50 KVBG']

In [70]:
z = "endgültig stillgelegt 2022 (ohne § 13b EnWG oder KVBG)"

In [71]:
def extract_state(state):
    if state == "In Betrieb":
        return "in Betrieb"
    elif "Endgültig Stillgelegt" in state:
        return "stillgelegt"
    elif "Vorläufig Stillgelegt" in state:
        return "vorläufig stillgelegt"
    elif "an Stilllegung gehindert" in state:
        return "Gesetzlich an Stilllegung gehindert"
    else:
        return state

In [72]:
def extract_state2(state):
    
    states = ['in Betrieb', 'Kapazitätsreserve', 'Netzreserve', 'Strommarktrückkehr', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt KVBG', 'stillgelegt']
    matches = ['In Betrieb', 'Kapazitätsreserve aufgrund von § 13e EnWG', 'Netzreserve aufgrund § 13b EnWG', 'befristete Strommarktrückkehr', 'vorläufig stillgelegt', r'endgültig stillgelegt 20[0-9]{2} \(nach § 13b EnWG\)', r'endgültig stillgelegt 20[0-9]{2} \(nach KVBG\)', r'[E|e]ndgültig [S|s]tillgelegt 20[0-9]{2} \([ohne]|[mit].']
    
    for s, m in list(zip(states, matches)):
        if re.search(m, state):
            return s
    
    return state

In [73]:
def extract_state3(state):
    
    states = ['in Betrieb', 'bnBm', 'KVBG', 'Kapazitätsreserve', 'Netzreserve', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt']
    matches = ['In Betrieb', 'besonderes netztechnisches Betriebsmittel', r'Netzreserve aufgrund von KVBG', 'Kapazitätsreserve aufgrund von § 13e EnWG', 'Netzreserve',  r'vorläufig stillgelegt', r'endgültig stillgelegt 20[0-9]{2} \(nach § 13b EnWG\)', r'stillgelegt']
    
    for s, m in list(zip(states, matches)):
        if re.search(m, state):
            return s
    
    return state

In [74]:
def extract_state4(state):
    
    states = ['in Betrieb', 'bnBm', 'KVBG', 'Kapazitätsreserve', 'Netzreserve', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt', 'stillgelegt']
    matches = ['In Betrieb', 'besonderes netztechnisches Betriebsmittel', r'Netzreserve aufgrund von KVBG', 'Kapazitätsreserve aufgrund von § 13e EnWG', 'Netzreserve',  r'vorläufig stillgelegt', r'endgültig stillgelegt 20[0-9]{2} \(nach § 13b EnWG\)', r'stillgelegt', r'Endgültig Stillgelegt 20[0-9]{2}']
    
    for s, m in list(zip(states, matches)):
        if re.search(m, state):
            return s
    
    return state

In [75]:
def extract_state5(state):
    
    states = ['in Betrieb', 'bnBm', 'KVBG', 'Kapazitätsreserve', 'Netzreserve', 'vorläufig stillgelegt', 'stillgelegt §13b EnWG', 'stillgelegt', 'stillgelegt', 'vorläufig stillgelegt']
    matches = ['InBetrieb', 'besonderes netztechnisches Betriebsmittel', r'Netzreserve aufgrund von KVBG', 'Kapazitätsreserve aufgrund von § 13e EnWG', 'Netzreserve',  r'vorläufig Stillgelegt', r'endgültig stillgelegt 20[0-9]{2} \(nach § 13b EnWG\)', r'stillgelegt', r'Endgültig Stillgelegt 20[0-9]{2}', 'zeitl. gestreckte Stilllegung aufgrund § 50 KVBG']
    
    for s, m in list(zip(states, matches)):
        if re.search(m, state):
            return s
    
    return state

In [76]:
sq = 'Endgültig Stillgelegt 2011 (ohne StA)'

In [77]:
extract_state5(sq)

'stillgelegt'

In [78]:
#pl4

In [79]:
pl4['company'] = pl4['company'].apply(lambda x: fix_company(x))
#pl4['endop'] = pl4['state'].apply(lambda x: extract_still(x))
pl4['state'] = pl4['state'].apply(lambda x: extract_state5(x))
pl4['chp'] = pl4['chp'].apply(lambda x: fix_kwk(x))

In [80]:
list(pl4.groupby("state").count().reset_index()['state'])

['KVBG',
 'Kapazitätsreserve',
 'Netzreserve',
 'bnBm',
 'in Betrieb',
 'stillgelegt',
 'vorläufig stillgelegt']

In [81]:
atest = pl4.groupby("state").count()

In [82]:
#atest

In [83]:
newdf = pl4.copy()

In [84]:
#newdf.dtypes

In [85]:
newdf["state"] = newdf["state"].astype('category')
newdf["state"] = newdf["state"].cat.set_categories(STATES, ordered=True)

In [86]:
newdf["chp"] = newdf["chp"].astype('category')
newdf["chp"] = newdf["chp"].cat.set_categories(['Ja', 'Nein'], ordered=True)

In [87]:
pl3 = newdf

In [88]:
atest = newdf.groupby(["power"]).count()
atest.sort_values("initialopYear", inplace=True, ascending=False)

In [89]:
#print(atest)

In [90]:
atest = pl3.groupby(["state"], observed=False).count()
# atest

In [91]:
# pl3.rename(columns={ pl3.columns[10]: "Energieträger"}, inplace=True)

In [92]:
#bm[bm.duplicated(['blockid'], keep=False)].sort_values("blockid", ascending=False)

In [93]:
#bm2 = bm.drop_duplicates()

In [94]:
#pl3

In [95]:
#bm

In [96]:
pl4 = bm.merge(pl3, how="right", left_on="sseid", right_on="blockid")

In [97]:
pl4.loc[pl4.blockid == "SEE966349705634"]

,InspireID_Betrieb,PRTR_Kennnummer,sseid,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO
316,NaN,NaN,NaN,Einzelanlage,SEE966349705634,EnBW AG,Heizkraftwerk Stuttgart-Münster GT 88,70376,Stuttgart,Voltastraße,45,Baden-Württemberg,Deutschland,2025.0,NaN,in Betrieb,Erdgas,Ja,Nein,61.5,61.0,GasturbinenmitAbhitzekessel,Hochspannung,Stuttgart Netze GmbH (SNB947592865054)


In [98]:
pl4

,InspireID_Betrieb,PRTR_Kennnummer,sseid,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO
0,SD666-16,666-16,SEE943690268513,stillgelegte Anlagen,SEE943690268513,NaN,Isar 2,84051,Essenbach,Dammstraße,NaN,Bayern,Deutschland,1988.0,2023.0,stillgelegt,Kernenergie,Nein,Nein,1485.00,1410.00,Druckwasserreaktor,NaN,NaN
1,SD666-14,666-14,SEE951462745445,stillgelegte Anlagen,SEE951462745445,NaN,Brokdorf,25576,Brokdorf,Osterende,NaN,Schleswig-Holstein,Deutschland,1986.0,2022.0,stillgelegt,Kernenergie,Nein,Nein,1480.00,1410.00,Druckwasserreaktor,NaN,NaN
2,SD666-12,666-12,BNA0802,stillgelegte Anlagen,BNA0802,NaN,Kernkraftwerk Philippsburg 2,76661,Philippsburg,Rheinschanzinsel,NaN,Baden-Württemberg,Deutschland,1985.0,2020.0,stillgelegt,Kernenergie,Nein,Nein,1468.00,1402.00,NaN,NaN,NaN
3,SD666-13,666-13,SEE930752846949,stillgelegte Anlagen,SEE930752846949,PreussenElektra,Grohnde,31860,Emmerthal,Kraftwerksgelände,NaN,Niedersachsen,Deutschland,1984.0,2022.0,stillgelegt,Kernenergie,Nein,Nein,1430.00,1360.00,Druckwasserreaktor,"Hochspannung, Hoechstspannung",Westfalen Weser Netz GmbH (SNB929881052512)
4,SD666-17,666-17,SEE944567587799,stillgelegte Anlagen,SEE944567587799,NaN,Emsland A,49811,Lingen,Am Hilgenberg,2,Niedersachsen,Deutschland,1988.0,2023.0,stillgelegt,Kernenergie,Nein,Nein,1406.00,1336.00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1509,NaN,NaN,NaN,stillgelegte Anlagen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Berlin,Deutschland,NaN,NaN,stillgelegt,Mineralölprodukte,NaN,Nein,0.03,0.03,NaN,NaN,NaN
1510,NaN,NaN,NaN,stillgelegte Anlagen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hamburg,Deutschland,NaN,NaN,stillgelegt,Mineralölprodukte,NaN,Nein,0.01,0.01,NaN,NaN,NaN
1511,NaN,NaN,NaN,stillgelegte Anlagen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Mecklenburg-Vorpommern,Deutschland,NaN,NaN,stillgelegt,Mineralölprodukte,NaN,Nein,0.00,0.00,NaN,NaN,NaN
1512,NaN,NaN,NaN,stillgelegte Anlagen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Baden-Württemberg,Deutschland,NaN,NaN,stillgelegt,Abfall,NaN,Nein,0.00,0.00,NaN,NaN,NaN


In [99]:
#pl4.loc[pd.isnull(pl4['sseid'])]

In [100]:
pl5 = pl4[pd.notnull(pl4['blockid'])]
#pl5 = pl4
pl5a = pl5[pd.notnull(pl5['power'])]
#pl6 =  pl5a.copy() #[pd.notnull(pl5a['plantid'])]
pl6 = pl5a.rename(columns={"InspireID_Betrieb": "plantid"})

In [101]:
pl6

,plantid,PRTR_Kennnummer,sseid,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO
0,SD666-16,666-16,SEE943690268513,stillgelegte Anlagen,SEE943690268513,NaN,Isar 2,84051,Essenbach,Dammstraße,NaN,Bayern,Deutschland,1988.0,2023.0,stillgelegt,Kernenergie,Nein,Nein,1485.000,1410.0,Druckwasserreaktor,NaN,NaN
1,SD666-14,666-14,SEE951462745445,stillgelegte Anlagen,SEE951462745445,NaN,Brokdorf,25576,Brokdorf,Osterende,NaN,Schleswig-Holstein,Deutschland,1986.0,2022.0,stillgelegt,Kernenergie,Nein,Nein,1480.000,1410.0,Druckwasserreaktor,NaN,NaN
2,SD666-12,666-12,BNA0802,stillgelegte Anlagen,BNA0802,NaN,Kernkraftwerk Philippsburg 2,76661,Philippsburg,Rheinschanzinsel,NaN,Baden-Württemberg,Deutschland,1985.0,2020.0,stillgelegt,Kernenergie,Nein,Nein,1468.000,1402.0,NaN,NaN,NaN
3,SD666-13,666-13,SEE930752846949,stillgelegte Anlagen,SEE930752846949,PreussenElektra,Grohnde,31860,Emmerthal,Kraftwerksgelände,NaN,Niedersachsen,Deutschland,1984.0,2022.0,stillgelegt,Kernenergie,Nein,Nein,1430.000,1360.0,Druckwasserreaktor,"Hochspannung, Hoechstspannung",Westfalen Weser Netz GmbH (SNB929881052512)
4,SD666-17,666-17,SEE944567587799,stillgelegte Anlagen,SEE944567587799,NaN,Emsland A,49811,Lingen,Am Hilgenberg,2,Niedersachsen,Deutschland,1988.0,2023.0,stillgelegt,Kernenergie,Nein,Nein,1406.000,1336.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1493,NaN,NaN,NaN,Einzelanlage,SEE962185382367,Papier- u. Kartonfabrik Varel GmbH & Co. KG,PKV Kraftwerk BGM 1,26316,Varel,Dangaster Straße,38,Niedersachsen,Deutschland,2006.0,NaN,in Betrieb,Biomasse,Ja,Ja,1.124,1.0,Verbrennungsmotor,Mittelspannung,EWE NETZ GmbH (SNB951051725711)
1494,NaN,NaN,NaN,Einzelanlage,SEE976753443961,Volkswagen AG,H78C Thermofunktionskanal Rollenprüfstand,38440,Wolfsburg,Berliner Ring,2,Niedersachsen,Deutschland,2017.0,NaN,in Betrieb,Mineralölprodukte,Nein,Nein,1.000,1.0,Verbrennungsmotor,Hochspannung,VW KRAFTWERK Gesellschaft mit beschränkter Haf...
1496,NaN,NaN,NaN,Einzelanlage,SEE945811865984,Caterpillar Energy Solutions,P13,68167,Mannheim,Carl-Benz-Straße,1,Baden-Württemberg,Deutschland,1939.0,NaN,in Betrieb,Erdgas,Nein,Nein,1.000,1.0,Verbrennungsmotor,Mittelspannung,MVV Netze GmbH (SNB985472799266)
1497,NaN,NaN,NaN,Einzelanlage,SEE970739914599,"Münchner Stadtentwässerung, Eigenbetrieb der L...",KLW1-KVA,80939,München,Freisinger Landstraße,187,Bayern,Deutschland,1998.0,NaN,in Betrieb,Biomasse,Nein,Ja,1.235,1.0,GegendruckmaschineohneEntnahme,Niederspannung,SWM Infrastruktur GmbH & Co. KG (SNB969473762610)


In [102]:
plnewtest = pl6.dropna(subset="plantid").sort_values("plantid")

In [103]:
pl6

,plantid,PRTR_Kennnummer,sseid,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO
0,SD666-16,666-16,SEE943690268513,stillgelegte Anlagen,SEE943690268513,NaN,Isar 2,84051,Essenbach,Dammstraße,NaN,Bayern,Deutschland,1988.0,2023.0,stillgelegt,Kernenergie,Nein,Nein,1485.000,1410.0,Druckwasserreaktor,NaN,NaN
1,SD666-14,666-14,SEE951462745445,stillgelegte Anlagen,SEE951462745445,NaN,Brokdorf,25576,Brokdorf,Osterende,NaN,Schleswig-Holstein,Deutschland,1986.0,2022.0,stillgelegt,Kernenergie,Nein,Nein,1480.000,1410.0,Druckwasserreaktor,NaN,NaN
2,SD666-12,666-12,BNA0802,stillgelegte Anlagen,BNA0802,NaN,Kernkraftwerk Philippsburg 2,76661,Philippsburg,Rheinschanzinsel,NaN,Baden-Württemberg,Deutschland,1985.0,2020.0,stillgelegt,Kernenergie,Nein,Nein,1468.000,1402.0,NaN,NaN,NaN
3,SD666-13,666-13,SEE930752846949,stillgelegte Anlagen,SEE930752846949,PreussenElektra,Grohnde,31860,Emmerthal,Kraftwerksgelände,NaN,Niedersachsen,Deutschland,1984.0,2022.0,stillgelegt,Kernenergie,Nein,Nein,1430.000,1360.0,Druckwasserreaktor,"Hochspannung, Hoechstspannung",Westfalen Weser Netz GmbH (SNB929881052512)
4,SD666-17,666-17,SEE944567587799,stillgelegte Anlagen,SEE944567587799,NaN,Emsland A,49811,Lingen,Am Hilgenberg,2,Niedersachsen,Deutschland,1988.0,2023.0,stillgelegt,Kernenergie,Nein,Nein,1406.000,1336.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1493,NaN,NaN,NaN,Einzelanlage,SEE962185382367,Papier- u. Kartonfabrik Varel GmbH & Co. KG,PKV Kraftwerk BGM 1,26316,Varel,Dangaster Straße,38,Niedersachsen,Deutschland,2006.0,NaN,in Betrieb,Biomasse,Ja,Ja,1.124,1.0,Verbrennungsmotor,Mittelspannung,EWE NETZ GmbH (SNB951051725711)
1494,NaN,NaN,NaN,Einzelanlage,SEE976753443961,Volkswagen AG,H78C Thermofunktionskanal Rollenprüfstand,38440,Wolfsburg,Berliner Ring,2,Niedersachsen,Deutschland,2017.0,NaN,in Betrieb,Mineralölprodukte,Nein,Nein,1.000,1.0,Verbrennungsmotor,Hochspannung,VW KRAFTWERK Gesellschaft mit beschränkter Haf...
1496,NaN,NaN,NaN,Einzelanlage,SEE945811865984,Caterpillar Energy Solutions,P13,68167,Mannheim,Carl-Benz-Straße,1,Baden-Württemberg,Deutschland,1939.0,NaN,in Betrieb,Erdgas,Nein,Nein,1.000,1.0,Verbrennungsmotor,Mittelspannung,MVV Netze GmbH (SNB985472799266)
1497,NaN,NaN,NaN,Einzelanlage,SEE970739914599,"Münchner Stadtentwässerung, Eigenbetrieb der L...",KLW1-KVA,80939,München,Freisinger Landstraße,187,Bayern,Deutschland,1998.0,NaN,in Betrieb,Biomasse,Nein,Ja,1.235,1.0,GegendruckmaschineohneEntnahme,Niederspannung,SWM Infrastruktur GmbH & Co. KG (SNB969473762610)


In [104]:
#plnewtest

In [105]:
#pl6.dtypes

In [106]:
pl6.loc[pl6.blockid == "BNA0711"]

,plantid,PRTR_Kennnummer,sseid,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO
200,NW300-0326774,06-05-300-0326774,BNA0711,stillgelegte Anlagen,BNA0711,NaN,Niederaußem,50129,Bergheim,NaN,NaN,Nordrhein-Westfalen,Deutschland,1963.0,2012.0,stillgelegt,Braunkohle,Nein,Nein,140.0,125.0,NaN,NaN,NaN


In [107]:
pl6['initialop'] = pl6['initialopYear'].apply(pd.to_numeric)
pl6['endop'] = pl6['endop'].apply(lambda x: to_year2(x))

In [108]:
#pl6.loc[pl6.blockid == "BNA0711"].dtypes

In [109]:
pl6.loc[pl6.blockid == "SEE999977738880"]

,plantid,PRTR_Kennnummer,sseid,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO,initialop
289,SD664-02,664-02,SEE999977738880,stillgelegte Anlagen,SEE999977738880,NaN,IKW Deuben,06682,Teuchern,Industriestraße,1,Sachsen-Anhalt,Deutschland,1936.0,2022.0,stillgelegt,Braunkohle,Ja,Nein,78.0,67.0,GegendruckmaschinemitEntnahme,NaN,NaN,1936.0


In [110]:
blocks = pl6.copy()
stammdaten = pl6.copy()

In [111]:
plants = blocks.copy()
#plants = plants.dropna(subset=["plantid", "initialop"]) #TODO: remove initialop from here

In [112]:
plants

,plantid,PRTR_Kennnummer,sseid,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO,initialop
0,SD666-16,666-16,SEE943690268513,stillgelegte Anlagen,SEE943690268513,NaN,Isar 2,84051,Essenbach,Dammstraße,NaN,Bayern,Deutschland,1988.0,2023.0,stillgelegt,Kernenergie,Nein,Nein,1485.000,1410.0,Druckwasserreaktor,NaN,NaN,1988.0
1,SD666-14,666-14,SEE951462745445,stillgelegte Anlagen,SEE951462745445,NaN,Brokdorf,25576,Brokdorf,Osterende,NaN,Schleswig-Holstein,Deutschland,1986.0,2022.0,stillgelegt,Kernenergie,Nein,Nein,1480.000,1410.0,Druckwasserreaktor,NaN,NaN,1986.0
2,SD666-12,666-12,BNA0802,stillgelegte Anlagen,BNA0802,NaN,Kernkraftwerk Philippsburg 2,76661,Philippsburg,Rheinschanzinsel,NaN,Baden-Württemberg,Deutschland,1985.0,2020.0,stillgelegt,Kernenergie,Nein,Nein,1468.000,1402.0,NaN,NaN,NaN,1985.0
3,SD666-13,666-13,SEE930752846949,stillgelegte Anlagen,SEE930752846949,PreussenElektra,Grohnde,31860,Emmerthal,Kraftwerksgelände,NaN,Niedersachsen,Deutschland,1984.0,2022.0,stillgelegt,Kernenergie,Nein,Nein,1430.000,1360.0,Druckwasserreaktor,"Hochspannung, Hoechstspannung",Westfalen Weser Netz GmbH (SNB929881052512),1984.0
4,SD666-17,666-17,SEE944567587799,stillgelegte Anlagen,SEE944567587799,NaN,Emsland A,49811,Lingen,Am Hilgenberg,2,Niedersachsen,Deutschland,1988.0,2023.0,stillgelegt,Kernenergie,Nein,Nein,1406.000,1336.0,NaN,NaN,NaN,1988.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1493,NaN,NaN,NaN,Einzelanlage,SEE962185382367,Papier- u. Kartonfabrik Varel GmbH & Co. KG,PKV Kraftwerk BGM 1,26316,Varel,Dangaster Straße,38,Niedersachsen,Deutschland,2006.0,NaN,in Betrieb,Biomasse,Ja,Ja,1.124,1.0,Verbrennungsmotor,Mittelspannung,EWE NETZ GmbH (SNB951051725711),2006.0
1494,NaN,NaN,NaN,Einzelanlage,SEE976753443961,Volkswagen AG,H78C Thermofunktionskanal Rollenprüfstand,38440,Wolfsburg,Berliner Ring,2,Niedersachsen,Deutschland,2017.0,NaN,in Betrieb,Mineralölprodukte,Nein,Nein,1.000,1.0,Verbrennungsmotor,Hochspannung,VW KRAFTWERK Gesellschaft mit beschränkter Haf...,2017.0
1496,NaN,NaN,NaN,Einzelanlage,SEE945811865984,Caterpillar Energy Solutions,P13,68167,Mannheim,Carl-Benz-Straße,1,Baden-Württemberg,Deutschland,1939.0,NaN,in Betrieb,Erdgas,Nein,Nein,1.000,1.0,Verbrennungsmotor,Mittelspannung,MVV Netze GmbH (SNB985472799266),1939.0
1497,NaN,NaN,NaN,Einzelanlage,SEE970739914599,"Münchner Stadtentwässerung, Eigenbetrieb der L...",KLW1-KVA,80939,München,Freisinger Landstraße,187,Bayern,Deutschland,1998.0,NaN,in Betrieb,Biomasse,Nein,Ja,1.235,1.0,GegendruckmaschineohneEntnahme,Niederspannung,SWM Infrastruktur GmbH & Co. KG (SNB969473762610),1998.0


In [113]:
plants["energysource"] = plants["energysource"].astype('category')
plants["energysource"] = plants["energysource"].cat.set_categories(ENERGIES, ordered=True)
plants["state"] = plants["state"].astype('category')
#plants["state"] = plants["state"].cat.set_categories(STATES, ordered=True)
plants["chp"] = plants["chp"].astype('category')
plants["chp"] = plants["chp"].cat.set_categories(['Ja', 'Nein'], ordered=True)
plants["federalstate"] = plants["federalstate"].astype('category')
plants["federalstate"] = plants["federalstate"].cat.set_categories(['Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen'])

In [114]:
pl1 = pl0.loc[pl0["energysource"].isin(["Kernenergie", "Erdgas", "Steinkohle", "Braunkohle", "Steinkohle"])]

In [115]:
p_grp = plants.groupby("plantid")

In [116]:
#plants.groupby("Energip_subgrper").max()

In [117]:
plants.groupby("state", observed=False).count()

,plantid,PRTR_Kennnummer,sseid,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO,initialop
state,,,,,,,,,,,,,,,,,,,,,,,,
in Betrieb,708,708,708,1127,1127,1127,1127,1127,1127,1111,1093,1127,1127,1127,0,1127,1127,1127,1127,1127,1127,1127,1127,1127
Kapazitätsreserve,12,12,12,16,16,16,16,16,16,16,15,16,16,16,0,16,16,16,16,16,16,16,16,16
Netzreserve,25,25,25,25,25,25,25,25,25,25,24,25,25,25,0,25,25,25,25,25,25,25,25,25
bnBm,14,14,14,14,14,14,14,14,14,14,14,14,14,14,0,14,14,14,14,14,14,14,14,14
KVBG,5,5,5,5,5,5,5,5,5,5,5,5,5,5,0,5,5,5,5,5,5,5,5,5
stillgelegt,209,209,209,249,249,28,249,249,249,185,125,249,249,248,249,249,249,249,249,249,132,28,28,248
vorläufig stillgelegt,11,11,11,15,15,15,15,15,15,15,13,15,15,15,0,15,15,15,15,15,15,15,15,15


In [118]:
ACTIVE_STATES

['in Betrieb',
 'Kapazitätsreserve',
 'Netzreserve',
 'bnBm',
 'vorläufig stillgelegt']

In [119]:
#plants.loc[plants['state'] == 'in Betrieb']

In [120]:
#plants.loc[plants['state'].isin(ACTIVE_STATES)].sort_values('power', ascending=False)

In [121]:
p_subgrp = plants.loc[plants['state'].isin(ACTIVE_STATES)].groupby('plantid') # TODO: add 

In [122]:
plants_act = pd.DataFrame()
for plantid, group in p_subgrp:
    entry = {}
    entry["plantid"] = plantid
    entry["activepower"] = group["power"].sum()
    #plants_act = plants_act.append(entry, ignore_index=True)
    plants_act = pd.concat([plants_act, pd.DataFrame([entry])], ignore_index=True)

In [123]:
plants_act

,plantid,activepower
0,06-02-B10117A007,239.000
1,BB16012651,1.957
2,BB16018798,176.000
3,BB23020389,4.700
4,BB23020490,333.500
...,...,...
268,ST18046,28.053
269,TH30013152,123.500
270,TH62013494,4.950
271,TH72012874,60.875


In [124]:
plants_a = pd.DataFrame()
for plantid, group in p_grp:
    #if not plantid == "06-05-300-9046797":
    #    continue
    entry = {}
    #print(entry)
    entry["plantid"] = plantid
    # entry["BlockID"] = group
    try:
        entry["plantname"] = group["plantname"].value_counts().index[0]
    except IndexError:
        entry["plantname"] = np.nan
    entry["federalstate"] = group["federalstate"].value_counts().index[0]
    entry["energysource"] = group["energysource"].min()
    try:
        entry["chp"] = group["chp"].min()
    except TypeError:
        #print("aneror")
        #print(plantid)
        #print(group['KWK'].count())
        entry["chp"] = ""
    entry["latestexpanded"] = group["initialop"].max()
    entry["initialop"] = group["initialop"].min()
    entry["totalpower"] = group["power"].sum()
    entry["state"] = group["state"].min()
    entry["blockcount"] = group["blockid"].count()
    try:
        entry["company"] = group["company"].value_counts().index[0]
    except IndexError:
        ;
    #print(entry)
    #break
    plants_a = pd.concat([plants_a, pd.DataFrame([entry])], ignore_index=True)

In [125]:
plants_a

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company
0,06-02-B10117A007,Tiefstack HKW Block 20,Hamburg,Steinkohle,Ja,2009.0,1993.0,239.000,in Betrieb,2,Hamburger Energiewerke GmbH
1,06-05-100-0030723,HKW Elberfeld,Nordrhein-Westfalen,Steinkohle,Ja,1992.0,1992.0,85.000,stillgelegt,1,NaN
2,06-05-100-0431554,KW Voerde,Nordrhein-Westfalen,Steinkohle,Nein,1985.0,1982.0,1390.000,stillgelegt,2,NaN
3,06-05-100-0853075,KW West,Nordrhein-Westfalen,Steinkohle,Nein,1971.0,1971.0,640.000,stillgelegt,2,NaN
4,16-30-31400103005,Heizkraftwerk Gera-Nord,Thüringen,Erdgas,Ja,1996.0,1996.0,74.000,stillgelegt,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...
328,ST18046,ZI Tr GT1,Sachsen-Anhalt,Erdgas,Ja,2017.0,1995.0,28.053,in Betrieb,6,K+S Minerals and Agriculture GmbH
329,TH30013152,DT,Thüringen,Erdgas,Ja,2022.0,1999.0,123.500,in Betrieb,5,SWE Energie GmbH
330,TH62013494,Kraftwerk UB GT1,Thüringen,Erdgas,Ja,2016.0,2016.0,4.950,in Betrieb,1,K+S Minerals and Agriculture GmbH
331,TH72012874,Gasmotor 5 Jena,Thüringen,Erdgas,Ja,2022.0,2022.0,60.875,in Betrieb,5,TEAG Thüringer Energie AG


In [126]:
int_list = ['blockcount', 'initialop', 'latestexpanded']
for column in int_list:
    plants_a[column] = plants_a[column].astype(int)
#plant['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)
#pl2['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)
#pl2['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)
#pl2['blockcount'] = pl2['blockcount'].apply(pd.to_numeric)

In [127]:
#plants_a.dtypes

In [128]:
plants_a["energysource"] = plants_a["energysource"].astype('category')
plants_a["energysource"] = plants_a["energysource"].cat.set_categories(ENERGIES, ordered=True)
plants_a["state"] = plants_a["state"].astype('category')
plants_a["state"] = plants_a["state"].cat.set_categories(STATES, ordered=True)
plants_a["chp"] = plants_a["chp"].astype('category')
plants_a["chp"] = plants_a["chp"].cat.set_categories(['Ja', 'Nein'], ordered=True)
plants_a["federalstate"] = plants_a["federalstate"].astype('category')
plants_a["federalstate"] = plants_a["federalstate"].cat.set_categories(['Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen'])

In [129]:
plants_a.loc[plants_a.plantid == "666-999"]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company


In [130]:
plants_a.loc[plants_a.plantid == "06-00176010435"]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company


In [131]:
plants_a.loc[plants_a.plantid == "06-05-900-0865327"]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company


In [132]:
#plants_a = plants.drop_duplicates(subset="KraftwerkID")

In [133]:
blocks.loc[blocks.sseid == "SEE930982693153"]

,plantid,PRTR_Kennnummer,sseid,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO,initialop
114,NW300-9046030,06-05-300-9046030,SEE930982693153,Einzelanlage,SEE930982693153,Knapsack Power GmbH & Co. KG,Knapsack I - Gasturbine GT11,50354,Hürth,Industriestraße,300,Nordrhein-Westfalen,Deutschland,2007.0,NaN,in Betrieb,Erdgas,Nein,Nein,295.0,295.0,GasturbinenmitnachgeschalteterDampfturbine,Hoechstspannung,Amprion GmbH (SNB976890256486),2007.0


In [134]:
column_titles = ['plantid', 'plantname', 'federalstate','energysource', 'chp', 'latestexpanded', 'initialop', 'totalpower', 'state', 'blockcount', 'company']
plants_b = plants_a.reindex(columns=column_titles)

In [135]:
blocks[blocks.duplicated(['blockid'], keep=False)].sort_values("blockid", ascending=False)

,plantid,PRTR_Kennnummer,sseid,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO,initialop


In [136]:
plants_b.sort_values("initialop", ascending=False, inplace=True)

In [137]:
plants_final = plants_b.loc[:]

In [138]:
#plants_final.dtypes

In [139]:
# plants_final

In [140]:
# stammdaten

In [141]:
#stammdaten

In [142]:
# cols = [0,2,3,9,10,11,12]
stammdaten = pl6.copy()
drop_list = ['plantid', 'energysource', 'initialop', 'chp', 'plantname', 'power', 'state', 'endop', 'company', 'sseid', 'TSO', 'voltagelevel']
stammdaten_dafuq = stammdaten.drop(drop_list, axis=1)
stammdaten = stammdaten_dafuq.copy()

In [143]:
stammdaten_dafuq.sort_values(['blockid'])

,PRTR_Kennnummer,datatype,blockid,plz,place,street,streetnum,federalstate,nation,initialopYear,eeg,grosspower,tech
544,06-08-4380932,stillgelegte Anlagen,BNA0011,79774,Albbruck,NaN,NaN,Baden-Württemberg,Deutschland,2009.0,Nein,28.0,NaN
1344,03-07-07244141350,stillgelegte Anlagen,BNA0012d,31061,Alfeld,Mühlenmarsch,NaN,Niedersachsen,Deutschland,1994.0,Nein,2.8,NaN
820,06-26200010633,stillgelegte Anlagen,BNA0059a,34225,Baunatal,NaN,NaN,Hessen,Deutschland,1961.0,Nein,13.2,NaN
170,06-11-01-1105607,stillgelegte Anlagen,BNA0075,12207,Berlin,Ostpreußendamm,NaN,Berlin,Deutschland,1972.0,Nein,150.0,NaN
171,06-11-01-1105607,stillgelegte Anlagen,BNA0076,12207,Berlin,Ostpreußendamm,NaN,Berlin,Deutschland,1974.0,Nein,150.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,NaN,Einzelanlage,SEE999819576335,39126,Magdeburg,Kraftwerk-Privatweg,7,Sachsen-Anhalt,Deutschland,2006.0,Nein,33.6,GegendruckmaschinemitEntnahme
1263,06-05-100-0154540,Einzelanlage,SEE999839188105,40476,Düsseldorf,Rather Straße,51,Nordrhein-Westfalen,Deutschland,2012.0,Nein,4.3,GasturbinenmitAbhitzekessel
19,06-02-BERZ003800,stillgelegte Anlagen,SEE999848168525,21079,Hamburg,Moorburger Schanze,2,Hamburg,Deutschland,2015.0,Nein,860.0,KondensationsmaschinemitEntnahme
426,NaN,Einzelanlage,SEE999966551940,14478,Potsdam,Zum Heizwerk,20,Brandenburg,Deutschland,1996.0,Nein,42.0,GasturbinenmitnachgeschalteterDampfturbine


In [144]:
stmp = pl6.copy()
stammdaten2 = stmp.merge(prtr, how="left", left_on="plantid", right_on="kennnummer")

In [145]:
#stammdaten2

In [146]:
DROP_ST = ['blockid', 'jahr', 'kennnummer','betriebsname','betriebsname_2','plz_x','ort','strasse','hausnr','bundesland','flusseinzugsgebiet','taet_nr','taetigkeit','activity','haupttaetigkeit','branche','sector','nace_id','nace_wirtschaftszweig','nace_sector','stoffgruppe','substances_group','schadstoff','pollutant','umweltkompartiment','releases_to','jahresfracht_freisetzung','versehentliche_freisetzung','schadstoff_schwellenwert','einheit','unit','bestimmungsmethode','determination_method','schutzgrund_fracht','confidential_reason_release','schutzgrund_betrieb','confidential_reason_facility']
DROP_ST2 = ['jahr', 'kennnummer','betriebsname','betriebsname_2','plz','ort','strasse','hausnr','bundesland','flusseinzugsgebiet','taet_nr','taetigkeit','activity','haupttaetigkeit','branche','sector','nace_id','nace_wirtschaftszweig','nace_sector','stoffgruppe','substances_group','schadstoff','pollutant','umweltkompartiment','releases_to','jahresfracht_freisetzung','versehentliche_freisetzung','schadstoff_schwellenwert','einheit','unit','bestimmungsmethode','determination_method','schutzgrund_fracht','confidential_reason_release','schutzgrund_betrieb','confidential_reason_facility']
DROP_ST3 = ['jahr', 'kennnummer', 'bundesland', 'flusseinzugsgebiet', "betriebsname", "betriebsname_2", 'taet_nr','taetigkeit','activity','haupttaetigkeit','branche','sector','nace_id','nace_wirtschaftszweig','nace_sector','stoffgruppe','substances_group','schadstoff','pollutant','umweltkompartiment','releases_to','jahresfracht_freisetzung','versehentliche_freisetzung','schadstoff_schwellenwert','einheit','unit','bestimmungsmethode','determination_method','schutzgrund_fracht','confidential_reason_release','schutzgrund_betrieb','confidential_reason_facility']


In [147]:
#DROP_ST = ['jahr']

In [148]:
plants_d1 = plants_final.merge(prtr, how="left", left_on="plantid", right_on="kennnummer")

In [149]:
#plants_d1.dtypes

In [150]:
pld2 = plants_d1.drop_duplicates(['plantid'])

In [151]:
pld3 = pld2.drop(DROP_ST3, axis=1)
plants_final1 = pld3

In [152]:
#plants_act

In [153]:
plants_final2 = plants_final1.merge(plants_act, on="plantid", how='left')

In [154]:
plants_final3 = plants_final2.dropna(subset=["plantid"])

In [155]:
plants_final3.groupby("energysource").count()

,plantid,plantname,federalstate,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower
energysource,,,,,,,,,,,,,,,,,,,
Kernenergie,8,8,8,8,8,8,8,8,8,1,0,0,0,0,0,0,0,0,0
Braunkohle,27,27,27,27,27,27,27,27,27,23,0,0,0,0,0,0,0,0,23
Steinkohle,61,61,61,61,61,61,61,61,61,42,4,4,4,4,4,4,4,4,41
Erdgas,216,216,216,216,216,216,216,216,216,197,1,1,1,1,1,1,1,1,195
Mineralölprodukte,21,21,21,21,21,21,21,21,21,15,0,0,0,0,0,0,0,0,14


In [156]:
plants_final2[pd.notnull(plants_final2['activepower'])].sort_values('activepower', ascending=False)

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower
220,SN70015796,Boxberg Block Q,Sachsen,Braunkohle,Ja,2012,1979,2470.000,in Betrieb,4,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2470.000
306,NW300-0326774,Niederaußem,Nordrhein-Westfalen,Braunkohle,Ja,2003,1963,3359.000,in Betrieb,8,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2220.000
266,NW100-0248923,Neurath F,Nordrhein-Westfalen,Braunkohle,Ja,2012,1972,4211.000,in Betrieb,7,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2120.000
211,BB45025564,Kraftwerk Jänschwalde Block F,Brandenburg,Braunkohle,Ja,1989,1981,3000.000,in Betrieb,6,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000.000
289,BWpf-450-2948214-00000000,GKM,Baden-Württemberg,Steinkohle,Ja,2015,1966,2363.000,in Betrieb,6,Grosskraftwerk Mannheim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1958.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25,SD661-89,BHKW Ford Saarlouis Modul 1,Saarland,Erdgas,Ja,2016,2016,4.322,in Betrieb,1,Ford-Werke GmbH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.322
18,SN60018636,BHKW-G66-KWK-klein,Sachsen,Erdgas,Ja,2017,2017,3.736,in Betrieb,1,Volkswagen Sachsen GmbH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.736
20,SD661-80,0050 - STW - BHKW - KWK,Nordrhein-Westfalen,Erdgas,Ja,2017,2017,3.313,in Betrieb,1,Stadtwerke Kempen GmbH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.313
23,NW700-0104479,BHKW-G,Nordrhein-Westfalen,Erdgas,Ja,2016,2016,1.999,in Betrieb,1,Westag AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.999


In [157]:
plants_final2.sort_values('activepower', ascending=False)

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower
220,SN70015796,Boxberg Block Q,Sachsen,Braunkohle,Ja,2012,1979,2470.0,in Betrieb,4,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2470.0
306,NW300-0326774,Niederaußem,Nordrhein-Westfalen,Braunkohle,Ja,2003,1963,3359.0,in Betrieb,8,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2220.0
266,NW100-0248923,Neurath F,Nordrhein-Westfalen,Braunkohle,Ja,2012,1972,4211.0,in Betrieb,7,RWE AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2120.0
211,BB45025564,Kraftwerk Jänschwalde Block F,Brandenburg,Braunkohle,Ja,1989,1981,3000.0,in Betrieb,6,LEAG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000.0
289,BWpf-450-2948214-00000000,GKM,Baden-Württemberg,Steinkohle,Ja,2015,1966,2363.0,in Betrieb,6,Grosskraftwerk Mannheim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1958.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,NW900-0884101,KW Lünen,Nordrhein-Westfalen,Steinkohle,Ja,1970,1962,473.0,stillgelegt,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
319,BWpf-450-1479296-00000000,Bestandsanlage_HKW_Aalen,Baden-Württemberg,Erdgas,Ja,1960,1960,15.0,stillgelegt,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
322,NW900-0271161,Shamrock,Nordrhein-Westfalen,Steinkohle,Ja,1957,1957,132.0,stillgelegt,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
324,NW100-0081105,Frimmersdorf,Nordrhein-Westfalen,Braunkohle,Nein,1970,1957,2008.0,stillgelegt,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [158]:
plants_final2[pd.isnull(plants_final2['plantid'])]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,company,betreiber,eigentuemer,plz,ort,strasse,hausnr,geo_lat_wgs84,geo_long_wgs84,activepower


In [159]:
stammdaten

,PRTR_Kennnummer,datatype,blockid,plz,place,street,streetnum,federalstate,nation,initialopYear,eeg,grosspower,tech
0,666-16,stillgelegte Anlagen,SEE943690268513,84051,Essenbach,Dammstraße,NaN,Bayern,Deutschland,1988.0,Nein,1485.000,Druckwasserreaktor
1,666-14,stillgelegte Anlagen,SEE951462745445,25576,Brokdorf,Osterende,NaN,Schleswig-Holstein,Deutschland,1986.0,Nein,1480.000,Druckwasserreaktor
2,666-12,stillgelegte Anlagen,BNA0802,76661,Philippsburg,Rheinschanzinsel,NaN,Baden-Württemberg,Deutschland,1985.0,Nein,1468.000,NaN
3,666-13,stillgelegte Anlagen,SEE930752846949,31860,Emmerthal,Kraftwerksgelände,NaN,Niedersachsen,Deutschland,1984.0,Nein,1430.000,Druckwasserreaktor
4,666-17,stillgelegte Anlagen,SEE944567587799,49811,Lingen,Am Hilgenberg,2,Niedersachsen,Deutschland,1988.0,Nein,1406.000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1493,NaN,Einzelanlage,SEE962185382367,26316,Varel,Dangaster Straße,38,Niedersachsen,Deutschland,2006.0,Ja,1.124,Verbrennungsmotor
1494,NaN,Einzelanlage,SEE976753443961,38440,Wolfsburg,Berliner Ring,2,Niedersachsen,Deutschland,2017.0,Nein,1.000,Verbrennungsmotor
1496,NaN,Einzelanlage,SEE945811865984,68167,Mannheim,Carl-Benz-Straße,1,Baden-Württemberg,Deutschland,1939.0,Nein,1.000,Verbrennungsmotor
1497,NaN,Einzelanlage,SEE970739914599,80939,München,Freisinger Landstraße,187,Bayern,Deutschland,1998.0,Ja,1.235,GegendruckmaschineohneEntnahme


In [160]:
#plants_final2.sort_values("activepower", ascending=False)

In [161]:
#stammdaten2.dtypes

In [162]:
#stammdaten3.sort_values(by=['plantid'], ascending=True)

In [163]:
#stammdaten3 = stammdaten2.drop(DROP_ST, axis=1)

In [164]:
#stammdaten4 = stammdaten3.drop_duplicates(['bnaid'])

In [165]:
#stammdaten4

In [166]:
#list(stammdaten2)

In [167]:
'''


"CREATE TABLE blocks(plantid TEXT, blockid TEXT NOT NULL PRIMARY KEY, blockdescription TEXT, federalstate TEXT, energysource TEXT, initialop INTEGER, chp TEXT,
blockname TEXT, netpower REAL, state TEXT, endop TEXT, company TEXT, FOREIGN KEY (blockid) REFERENCES addresses(blockid) ON DELETE CASCADE);"
662-01|BNA0164|Vattenfall GmbH|Brunsbüttel|Schleswig-Holstein|GT D|0.0|stillgelegt|Mineralölprodukte|Nein|63.5|201

'''


'\n\n\n"CREATE TABLE blocks(plantid TEXT, blockid TEXT NOT NULL PRIMARY KEY, blockdescription TEXT, federalstate TEXT, energysource TEXT, initialop INTEGER, chp TEXT,\nblockname TEXT, netpower REAL, state TEXT, endop TEXT, company TEXT, FOREIGN KEY (blockid) REFERENCES addresses(blockid) ON DELETE CASCADE);"\n662-01|BNA0164|Vattenfall GmbH|Brunsbüttel|Schleswig-Holstein|GT D|0.0|stillgelegt|Mineralölprodukte|Nein|63.5|201\n\n'

In [168]:
#|664-02|IKW

In [169]:
blocks2 = blocks[['blockid', 'plantid', 'plantname', 'federalstate', 'energysource', 'initialop', 'chp', 'power', 'state', 'endop', 'company']]

In [170]:
blocks2['initialop'] = blocks2['initialop'].astype('Int32')
blocks2['endop'] = blocks2['endop'].astype('Int32')

In [171]:
blocks2.loc[blocks2.blockid == "06-08-2948214"]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company


In [172]:
blocks2.loc[blocks2.plantid == "BYS00041"]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
96,SEE952372080091,BYS00041,SWM HKW Nord 2 T20,Bayern,Erdgas,1991,Ja,333.0,in Betrieb,<NA>,SWM Services GmbH
604,SEE998278854237,BYS00041,SWM HKW Nord 3 T30,Bayern,Abfall,1984,Ja,22.0,in Betrieb,<NA>,SWM Services GmbH
676,SEE942366584926,BYS00041,SWM HKW Nord 1 T10,Bayern,Abfall,1991,Ja,18.0,in Betrieb,<NA>,SWM Services GmbH


In [173]:
blocks2['chp'] = blocks2['chp'].fillna('Nein')

In [174]:
list(blocks2)

['blockid',
 'plantid',
 'plantname',
 'federalstate',
 'energysource',
 'initialop',
 'chp',
 'power',
 'state',
 'endop',
 'company']

In [175]:
stammdaten['plz'] = stammdaten['plz'].astype('Int32')

In [176]:
stammdaten.loc[stammdaten.duplicated(subset=['blockid'])]

,PRTR_Kennnummer,datatype,blockid,plz,place,street,streetnum,federalstate,nation,initialopYear,eeg,grosspower,tech


In [177]:
stammdaten.drop_duplicates(subset="blockid", inplace=True)
blocks2.drop_duplicates(subset="blockid", inplace=True)
plants_final2.drop_duplicates(subset="plantid", inplace=True)

In [178]:
stammdaten2.loc[stammdaten2.blockid == "SEE915851127786"]

,plantid,PRTR_Kennnummer,sseid,datatype,blockid,company,plantname,plz_x,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO,initialop,jahr,kennnummer,betriebsname,betriebsname_2,betreiber,eigentuemer,plz_y,ort,strasse,hausnr,bundesland,flusseinzugsgebiet,geo_lat_wgs84,geo_long_wgs84,taet_nr,taetigkeit,activity,haupttaetigkeit,branche,sector,nace_id,nace_wirtschaftszweig,nace_sector,stoffgruppe,substances_group,schadstoff,pollutant,umweltkompartiment,releases_to,jahresfracht_freisetzung,versehentliche_freisetzung,schadstoff_schwellenwert,einheit,unit,bestimmungsmethode,determination_method,schutzgrund_fracht,confidential_reason_release,schutzgrund_betrieb,confidential_reason_facility
884,NaN,NaN,NaN,Einzelanlage,SEE915851127786,STAWAG – Stadt- und Städteregionswerke Aachen AG,BHKW Schwarzer Weg,52070,Aachen,Schwarzer Weg,19 b,Nordrhein-Westfalen,Deutschland,2023.0,NaN,in Betrieb,Erdgas,Ja,Nein,22.535,21.505,Verbrennungsmotor,Mittelspannung,Regionetz GmbH (SNB911641710114),2023.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [179]:
stammdaten2 = stammdaten[['blockid', 'plz', 'place', 'street', 'streetnum', 'federalstate']]

In [180]:
blocks2.loc[blocks2.blockid == "SEE966349705634"]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
316,SEE966349705634,NaN,Heizkraftwerk Stuttgart-Münster GT 88,Baden-Württemberg,Erdgas,2025,Ja,61.0,in Betrieb,<NA>,EnBW AG


In [181]:
blocks2.loc[blocks2.blockid == "SEE913516809497"]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
11,SEE913516809497,NW500-0915123,Datteln 4,Nordrhein-Westfalen,Steinkohle,2020,Ja,1052.0,in Betrieb,<NA>,Datteln 4 a.s & Co. KG


In [182]:
#blocks2.loc['C', 'x'] = "BNA1949"

In [183]:
stammdaten2.to_csv("stammdaten_nh_new.csv", index=False, header=False)
blocks2.to_csv("blocks_nh_2.csv", index=False, header=False)
blocks2.to_csv("blocks_new_nh.csv", index=False, header=False)
plants_final2.to_csv("plants_nh_2.csv", index=False, header=False)
stammdaten2.to_csv("stammdaten.csv", index=False)
blocks2.to_csv("blocks_2.csv", index=False)
plants_final2.to_csv("plants_2.csv", index=False)

In [184]:
#sqlite3 plantwatch.db  "CREATE TABLE plants(plantid TEXT NOT NULL PRIMARY KEY, plantname TEXT, federalstate TEXT, energysource TEXT, chp TEXT, latestexpanded INT, initialop INT, totalpower REAL, state TEXT, blockcount INT,  company TEXT, plz TEXT, place TEXT, street TEXT, number TEXT, latitude REAL, longitude REAL, activepower REAL, energy_2015 INTEGER,  energy_2016 INTEGER,  energy_2017 INTEGER,  energy_2018 INTEGER,  energy_2019 INTEGER, energy_2020 INTEGER, energy_2021 INTEGER, co2_2007 INTEGER,  co2_2008 INTEGER,  co2_2009 INTEGER,  co2_2010 INTEGER,  co2_2011 INTEGER,  co2_2012 INTEGER,  co2_2013 INTEGER,  co2_2014 INTEGER,  co2_2015 INTEGER,  co2_2016 INTEGER,  co2_2017 INTEGER,  co2_2018 INTEGER, co2_2019 INTEGER, co2_2020 INTEGER, FOREIGN KEY (plantid) REFERENCES blocks(plantid) ON DELETE CASCADE);"


In [185]:
blocks2.dropna(subset=['plantid'])

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
0,SEE943690268513,SD666-16,Isar 2,Bayern,Kernenergie,1988,Nein,1410.000,stillgelegt,2023,NaN
1,SEE951462745445,SD666-14,Brokdorf,Schleswig-Holstein,Kernenergie,1986,Nein,1410.000,stillgelegt,2022,NaN
2,BNA0802,SD666-12,Kernkraftwerk Philippsburg 2,Baden-Württemberg,Kernenergie,1985,Nein,1402.000,stillgelegt,2020,NaN
3,SEE930752846949,SD666-13,Grohnde,Niedersachsen,Kernenergie,1984,Nein,1360.000,stillgelegt,2022,PreussenElektra
4,SEE944567587799,SD666-17,Emsland A,Niedersachsen,Kernenergie,1988,Nein,1336.000,stillgelegt,2023,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1469,SEE965327731764,NW300-0907015,BHKW Jülich,Nordrhein-Westfalen,Erdgas,2002,Ja,1.387,in Betrieb,<NA>,Pfeifer & Langen GmbH & Co. KG
1474,SEE976944273018,SD661-85,"Rollenprüfstand 1, Gebäude 38",Rheinland-Pfalz,Mineralölprodukte,2019,Nein,1.360,in Betrieb,<NA>,Daimler Truck AG
1477,SEE947712976042,HE50002124,Kraftwerk NE GT KW,Hessen,Erdgas,1990,Ja,1.214,in Betrieb,<NA>,K+S Minerals and Agriculture GmbH
1480,SEE912440649056,NW300-0072412,Euskirchen Generator 3,Nordrhein-Westfalen,Braunkohle,1980,Ja,1.133,stillgelegt,2022,Pfeifer & Langen GmbH & Co. KG


In [186]:
blocks2.shape

(1451, 11)

In [187]:
blocks2.dropna(subset=['plantid']).shape

(984, 11)

In [188]:
#stammdaten.dropna(subset=['sseid', 'bnaid']).sort_values(by=['sseid'], ascending=False)

In [189]:
stammdaten

,PRTR_Kennnummer,datatype,blockid,plz,place,street,streetnum,federalstate,nation,initialopYear,eeg,grosspower,tech
0,666-16,stillgelegte Anlagen,SEE943690268513,84051,Essenbach,Dammstraße,NaN,Bayern,Deutschland,1988.0,Nein,1485.000,Druckwasserreaktor
1,666-14,stillgelegte Anlagen,SEE951462745445,25576,Brokdorf,Osterende,NaN,Schleswig-Holstein,Deutschland,1986.0,Nein,1480.000,Druckwasserreaktor
2,666-12,stillgelegte Anlagen,BNA0802,76661,Philippsburg,Rheinschanzinsel,NaN,Baden-Württemberg,Deutschland,1985.0,Nein,1468.000,NaN
3,666-13,stillgelegte Anlagen,SEE930752846949,31860,Emmerthal,Kraftwerksgelände,NaN,Niedersachsen,Deutschland,1984.0,Nein,1430.000,Druckwasserreaktor
4,666-17,stillgelegte Anlagen,SEE944567587799,49811,Lingen,Am Hilgenberg,2,Niedersachsen,Deutschland,1988.0,Nein,1406.000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1493,NaN,Einzelanlage,SEE962185382367,26316,Varel,Dangaster Straße,38,Niedersachsen,Deutschland,2006.0,Ja,1.124,Verbrennungsmotor
1494,NaN,Einzelanlage,SEE976753443961,38440,Wolfsburg,Berliner Ring,2,Niedersachsen,Deutschland,2017.0,Nein,1.000,Verbrennungsmotor
1496,NaN,Einzelanlage,SEE945811865984,68167,Mannheim,Carl-Benz-Straße,1,Baden-Württemberg,Deutschland,1939.0,Nein,1.000,Verbrennungsmotor
1497,NaN,Einzelanlage,SEE970739914599,80939,München,Freisinger Landstraße,187,Bayern,Deutschland,1998.0,Ja,1.235,GegendruckmaschineohneEntnahme


In [190]:
blocks.loc[blocks["blockid"] == "BNA0645"]

,plantid,PRTR_Kennnummer,sseid,datatype,blockid,company,plantname,plz,place,street,streetnum,federalstate,nation,initialopYear,endop,state,energysource,chp,eeg,grosspower,power,tech,voltagelevel,TSO,initialop


In [191]:
#bm2.to_csv("bpm.csv", index=False)

In [192]:
#blocks

In [193]:
#blocks.groupby('Energieträger').count()

In [194]:
#pl4 = blocks.loc[blocks['Energieträger'] == "Mineralölprodukte"]

In [195]:
#pl4

In [196]:
#pd.set_option('display.max_rows', None)

In [197]:
#pl4.loc[pl4['Energieträger'] == "Mineralölprodukte"].sort_values(["Bundesland", "Unternehmen", "Kraftwerksname"])

In [198]:
#plants_final2

In [199]:
blocks2.sort_values('power', ascending=False)[0:20]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
0,SEE943690268513,SD666-16,Isar 2,Bayern,Kernenergie,1988,Nein,1410.0,stillgelegt,2023,NaN
1,SEE951462745445,SD666-14,Brokdorf,Schleswig-Holstein,Kernenergie,1986,Nein,1410.0,stillgelegt,2022,NaN
2,BNA0802,SD666-12,Kernkraftwerk Philippsburg 2,Baden-Württemberg,Kernenergie,1985,Nein,1402.0,stillgelegt,2020,NaN
3,SEE930752846949,SD666-13,Grohnde,Niedersachsen,Kernenergie,1984,Nein,1360.0,stillgelegt,2022,PreussenElektra
4,SEE944567587799,SD666-17,Emsland A,Niedersachsen,Kernenergie,1988,Nein,1336.0,stillgelegt,2023,NaN
5,SEE985577062814,SD666-18,GKN II,Baden-Württemberg,Kernenergie,1989,Nein,1310.0,stillgelegt,2023,NaN
6,SEE927528071629,SD666-11,Gundremmingen C,Bayern,Kernenergie,1984,Nein,1288.0,stillgelegt,2022,NaN
7,BNA0381,SD666-11,Kernkraft Gundremmingen,Bayern,Kernenergie,1984,Nein,1284.0,stillgelegt,2018,NaN
8,BNA0355,SD666-15,Grafenrheinfeld,Bayern,Kernenergie,1982,Nein,1275.0,stillgelegt,2015,NaN
9,SEE925599434282,NW100-0248923,Neurath F,Nordrhein-Westfalen,Braunkohle,2012,Ja,1060.0,in Betrieb,<NA>,RWE AG


In [200]:
blocks2.sort_values('power', ascending=False)[0:20]

,blockid,plantid,plantname,federalstate,energysource,initialop,chp,power,state,endop,company
0,SEE943690268513,SD666-16,Isar 2,Bayern,Kernenergie,1988,Nein,1410.0,stillgelegt,2023,NaN
1,SEE951462745445,SD666-14,Brokdorf,Schleswig-Holstein,Kernenergie,1986,Nein,1410.0,stillgelegt,2022,NaN
2,BNA0802,SD666-12,Kernkraftwerk Philippsburg 2,Baden-Württemberg,Kernenergie,1985,Nein,1402.0,stillgelegt,2020,NaN
3,SEE930752846949,SD666-13,Grohnde,Niedersachsen,Kernenergie,1984,Nein,1360.0,stillgelegt,2022,PreussenElektra
4,SEE944567587799,SD666-17,Emsland A,Niedersachsen,Kernenergie,1988,Nein,1336.0,stillgelegt,2023,NaN
5,SEE985577062814,SD666-18,GKN II,Baden-Württemberg,Kernenergie,1989,Nein,1310.0,stillgelegt,2023,NaN
6,SEE927528071629,SD666-11,Gundremmingen C,Bayern,Kernenergie,1984,Nein,1288.0,stillgelegt,2022,NaN
7,BNA0381,SD666-11,Kernkraft Gundremmingen,Bayern,Kernenergie,1984,Nein,1284.0,stillgelegt,2018,NaN
8,BNA0355,SD666-15,Grafenrheinfeld,Bayern,Kernenergie,1982,Nein,1275.0,stillgelegt,2015,NaN
9,SEE925599434282,NW100-0248923,Neurath F,Nordrhein-Westfalen,Braunkohle,2012,Ja,1060.0,in Betrieb,<NA>,RWE AG
